In [1]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import nltk
import re

# nltk.download('stopwords')
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('wordnet')
# For Medium the row indexes : [2, 8, 10, 31]
pd.set_option('display.max_colwidth', None)


In [ ]:
def preprocess_sentiment_data(data):
    """
    Preprocess the sentiment data by:
    1. Mapping polarity values to sentiment labels (0 for negative, 1 for positive).
    2. Dropping rows where 'title' or 'text' is missing or empty.
    3. Returning the preprocessed dataframe with clean columns for further processing.

    Parameters:
    - data (pd.DataFrame): Input dataframe with 'polarity', 'title', and 'text' columns.

    Returns:
    - pd.DataFrame: Preprocessed dataframe with 'title', 'text', and 'sentiment' columns.
    """
    # Map polarity values to sentiment labels
    data['sentiment'] = data['polarity'].map({1: 0, 2: 1})
    
    # Drop rows where 'title' or 'text' is missing or empty
    data = data.dropna(subset=['title', 'text'])  # Drop rows where title or text is NaN
    data = data[data['title'].str.strip() != ""]  # Remove rows with empty title
    data = data[data['text'].str.strip() != ""]   # Remove rows with empty text
    
    data = data.drop_duplicates(subset=['title', 'text'])

    # Drop unnecessary columns
    data = data.drop(columns=["polarity"])
    
    return data

In [ ]:
from nltk.stem import WordNetLemmatizer
from contractions import fix  # pip install contractions
from nltk.corpus import wordnet

class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X.dropna(subset=self.columns, inplace=True)

        for col in self.columns:
            X[col] = X[col].apply(self.expand_contractions)
            X[col] = X[col].apply(self.lowercase_text)
            X[col] = X[col].apply(self.remove_punctuation)
            X[col] = X[col].apply(self.handle_negations)
            X[col] = X[col].apply(self.tokenize_text)
            X[col] = X[col].apply(self.remove_stopwords)
            
            # **New Step: POS Tagging**
            X[col] = X[col].apply(self.pos_tag_tokens)
            
            # **Modified Lemmatization with POS:**
            X[col] = X[col].apply(self.lemmatize_with_pos)
            X[col] = X[col].apply(self.combine_tokens)
            
        X.rename(columns={col: f"processed_{col}" for col in self.columns}, inplace=True)

        for col in self.columns:
            X = X[X[f"processed_{col}"].str.strip() != ""]

        return X

    def lowercase_text(self, text):
        return text.lower()
    
    def expand_contractions(self, text):
        return fix(text)

    def handle_negations(self, text):
        # After contractions are expanded, we mostly deal with words like "not", "no", "never", etc.
        negation_words = "not|never|no|none|nothing|nowhere|nobody|neither|nor"
        pattern = rf"\b({negation_words})\b\s+(\w+)"
        return re.sub(pattern, r"\1_\2", text)

    def remove_punctuation(self, text):
        return re.sub(r'[^a-z\s]', '', text)

    def tokenize_text(self, text):
        return word_tokenize(text)

    def remove_stopwords(self, tokens):
        return [word for word in tokens if word not in self.stop_words]

    def pos_tag_tokens(self, tokens):
        return nltk.pos_tag(tokens)  # Returns list of tuples: [(token, pos_tag), ...]

    def get_wordnet_pos(self, treebank_tag):
        # Convert Penn Treebank tag to a WordNet POS
        if treebank_tag.startswith('J'):
            return wordnet.ADJ
        elif treebank_tag.startswith('V'):
            return wordnet.VERB
        elif treebank_tag.startswith('N'):
            return wordnet.NOUN
        elif treebank_tag.startswith('R'):
            return wordnet.ADV
        else:
            return wordnet.NOUN

    def lemmatize_with_pos(self, pos_tagged_tokens):
        # pos_tagged_tokens is a list of (word, pos) tuples
        lemmatized = []
        for word, tag in pos_tagged_tokens:
            wn_tag = self.get_wordnet_pos(tag)
            lemmatized.append(self.lemmatizer.lemmatize(word, wn_tag))
        return lemmatized

    def combine_tokens(self, tokens):
        return ' '.join(tokens)



In [ ]:
train_path = f"/Users/mertarcan/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2/train.csv"
test_path = f"/Users/mertarcan/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2/test.csv"

raw_data = pd.read_csv(train_path, header=None, names=['polarity', 'title', 'text'])
df = preprocess_sentiment_data(raw_data) 


In [ ]:
preprocessor = TextPreprocessor(columns=['title', 'text'])
df_preprocessed = preprocessor.fit_transform(df)
print(df_preprocessed)

In [ ]:
output_path = 'preprocessed_data.csv'
df_preprocessed.to_csv(output_path, index=False)
print(f"Preprocessed data saved to {output_path}")

In [ ]:
df_preprocessed

## 3. Vectorizing the Text

In [ ]:
# Load data If necessary:
# df_preprocessed = pd.read_csv("preprocessed_data.csv")

In [ ]:
from sklearn.model_selection import train_test_split
# Define features and labels
X = df_preprocessed[['processed_title', 'processed_text']]  # Use both title and text columns
y = df_preprocessed['sentiment']  # Sentiment labels

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


In [ ]:

# Baseline Logistic Regression
baseline_lr = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('title', TfidfVectorizer(max_features=1000, min_df=1), 'processed_title'),
        ('text', TfidfVectorizer(max_features=10000, min_df=1), 'processed_text')
    ], transformer_weights={'title': 3, 'text': 1})),
    ('model', LogisticRegression(solver='saga'))
])
baseline_lr.fit(X_train, y_train)
baseline_lr_score = baseline_lr.score(X_val, y_val)

# Baseline XGBoost
baseline_xgb = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('title', TfidfVectorizer(max_features=1000, min_df=1), 'processed_title'),
        ('text', TfidfVectorizer(max_features=10000, min_df=1), 'processed_text')
    ], transformer_weights={'title': 3, 'text': 1})),
    ('model', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'))
])
baseline_xgb.fit(X_train, y_train)
baseline_xgb_score = baseline_xgb.score(X_val, y_val)

# Compare baseline scores and choose one to fine-tune


In [39]:
TITLE_WEIGHT = 1
TEXT_WEIGHT = 1

myVectorizer = ColumnTransformer(
    transformers=[
        ('title', TfidfVectorizer(max_features=2000, min_df=2), 'processed_title'),
        ('text', TfidfVectorizer(max_features=20000, min_df=2), 'processed_text')
    ],
    transformer_weights={'title': TITLE_WEIGHT, 'text': TEXT_WEIGHT}
)

X_train_vectorized = myVectorizer.fit_transform(X_train)
# Transform the validation data with the already fitted transformer
X_val_vectorized = myVectorizer.transform(X_val)

In [ ]:


model = LogisticRegression(solver='saga', max_iter=1000)
model.fit(X_train_vectorized, y_train)


Training accuracy: 0.9163306718733656


In [41]:
training_score = model.score(X_train_vectorized, y_train)
validation_score = model.score(X_val_vectorized, y_val)
print("Training accuracy:", training_score)
print("Validation accuracy:", validation_score)

Training accuracy: 0.9188883779035367
Validation accuracy: 0.9163306718733656


In [ ]:

# Define weights for title and text
TITLE_WEIGHT = 1
TEXT_WEIGHT = 1

# Define TF-IDF vectorizers for title and text
preprocessor = ColumnTransformer(
    transformers=[
        ('title', TfidfVectorizer(), 'processed_title'),  # Title vectorizer
        ('text', TfidfVectorizer(), 'processed_text')     # Text vectorizer
    ],
    transformer_weights={'title': TITLE_WEIGHT, 'text': TEXT_WEIGHT}
)

# Define pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())  # Placeholder for grid search
])

param_grid = [
    # If Logistic Regression turns out better
    {
        # Keep feature set simpler
        'preprocessor__title__max_features': [10000, 20000, 30000],
        'preprocessor__text__max_features': [100000, 200000, 3000000],
        # Keep weighting scenarios minimal
        'preprocessor__transformer_weights': [
            {'title': 1, 'text': 1},
            {'title': 2, 'text': 1},
        ],
        # Keep min_df consistent to limit vocabulary size and noise
        'preprocessor__title__min_df': [2],
        'preprocessor__text__min_df': [2],
        'model': [LogisticRegression(solver='saga', max_iter=1000)],
        # Limit C values to a small range
        'model__C': [1]
    },
]




In [ ]:
grid_search = GridSearchCV(pipeline, param_grid, cv=3, verbose=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)


In [ ]:
#Best parameters found: {'model': LogisticRegression(max_iter=1000, solver='saga'), 'model__C': 1, 'preprocessor__text__max_features': 100000, 'preprocessor__text__min_df': 2, 'preprocessor__title__max_features': 20000, 'preprocessor__title__min_df': 2, 'preprocessor__transformer_weights': {'title': 1, 'text': 1}}
#Best score: 0.9141279984273099

In [ ]:
import pickle
import json

# Suppose `best_params` is a dictionary of your best found parameters
best_params = {'model': LogisticRegression(max_iter=1000, solver='saga'), 'model__C': 1, 'preprocessor__text__max_features': 100000, 'preprocessor__text__min_df': 2, 'preprocessor__title__max_features': 20000, 'preprocessor__title__min_df': 2, 'preprocessor__transformer_weights': {'title': 1, 'text': 1}}

with open('best_params.pkl', 'wb') as f:
    pickle.dump(best_params, f)

print("Best parameters saved to best_params.pkl")



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Best hyperparameters from grid search
TITLE_WEIGHT = 1
TEXT_WEIGHT = 1
TITLE_MAX_FEATURES = 20000
TEXT_MAX_FEATURES = 100000
MODEL_C = 1
MODEL_SOLVER = 'saga'



# Define TF-IDF vectorizers for title and text with best hyperparameters
preprocessor = ColumnTransformer(
    transformers=[
        ('title', TfidfVectorizer(max_features=TITLE_MAX_FEATURES), 'processed_title'),
        ('text', TfidfVectorizer(max_features=TEXT_MAX_FEATURES, min_df=2), 'processed_text')
    ],
    transformer_weights={'title': TITLE_WEIGHT, 'text': TEXT_WEIGHT}
)

# Define the Logistic Regression model with best hyperparameters
model = LogisticRegression(C=MODEL_C, solver=MODEL_SOLVER)

# Define the pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

with open('best_params.pkl', 'rb') as f:
    loaded_params = pickle.load(f)

pipeline.set_params(**loaded_params)


# Assuming X is your features DataFrame and y is your target variable
# Fit the pipeline to your data
print(pipeline.get_params())


In [ ]:
import joblib

pipeline.fit(X_train, y_train)

# Save the pipeline to a file
joblib.dump(pipeline, 'trained_pipeline.pkl')


In [ ]:
# Access the trained Logistic Regression model
trained_model = pipeline.named_steps['model']


In [ ]:
# Access model coefficients
coefficients = trained_model.coef_
intercept = trained_model.intercept_

print("Coefficients:", coefficients)
print("Intercept:", intercept)


In [ ]:
# Import necessary libraries for evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

def eval(X_val, y_val, pipeline):
    # Make predictions on the validation set
    predictions = pipeline.predict(X_val)

    # Calculate accuracy
    accuracy = accuracy_score(y_val, predictions)
    print(f'Accuracy: {accuracy:.4f}')

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_val, predictions)
    print('Confusion Matrix:')
    print(conf_matrix)

    # Classification report (Precision, Recall, F1-Score)
    class_report = classification_report(y_val, predictions)
    print('Classification Report:')
    print(class_report)

    # Calculate ROC AUC Score
    # For binary classification, get probability estimates for the positive class
    y_probs = pipeline.predict_proba(X_val)[:, 1]
    roc_auc = roc_auc_score(y_val, y_probs)
    print(f'ROC AUC Score: {roc_auc:.4f}')

    # Plot ROC Curve
    fpr, tpr, thresholds = roc_curve(y_val, y_probs)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})', color='blue', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', linewidth=2)  # Diagonal line representing random chance
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()


In [ ]:
eval(X_val= X_val, y_val = y_val, pipeline = pipeline)

In [ ]:
# Get feature names from the vectorizers
title_features = pipeline.named_steps['preprocessor'].named_transformers_['title'].get_feature_names_out()
text_features = pipeline.named_steps['preprocessor'].named_transformers_['text'].get_feature_names_out()

# Combine feature names
all_features = list(title_features) + list(text_features)

# Get coefficients from the model
coefficients = trained_model.coef_[0]

# Create a DataFrame to display feature importance
import pandas as pd
feature_importance = pd.DataFrame({
    'Feature': all_features,
    'Coefficient': coefficients
})

# Sort by absolute value of coefficients
feature_importance['AbsCoefficient'] = feature_importance['Coefficient'].abs()
feature_importance.sort_values(by='AbsCoefficient', ascending=False, inplace=True)

# Display top 10 features
print(feature_importance.head(10))
